# Dhruv Thumar | 3753860
## EDA


In [1]:
pip install gdown


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

#settings to display numbers nice on the display
pd.options.display.float_format = '{:20.2f}'.format

#show all columns on the output
pd.set_option('display.max_columns', 999)

In [3]:

# Load the dataset
file_id = '1VflPJgqBUumgu5K1S6RmyG-jxmqMjGna'
output_filename = 'downloaded_data.csv' 

import gdown
gdown.download(id=file_id, output=output_filename, quiet=False)

df = pd.read_csv(output_filename)

Downloading...
From: https://drive.google.com/uc?id=1VflPJgqBUumgu5K1S6RmyG-jxmqMjGna
To: c:\Users\dmthu\OneDrive - University of New Brunswick\UNB\Winter '26\CS4403\Project_CS4403-retail-mining\downloaded_data.csv
100%|██████████| 46.1M/46.1M [00:07<00:00, 6.04MB/s]


# Data Exploration

In [4]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850.00,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,01-12-2010 08:26,7.65,17850.00,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,01-12-2010 08:26,4.25,17850.00,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,01-12-2010 08:28,1.85,17850.00,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,01-12-2010 08:28,1.85,17850.00,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,01-12-2010 08:34,1.69,13047.00,United Kingdom


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [6]:
#convert invoice date to datetime64[ns]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

In [7]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.00,232959,541909.00,406829.00
mean,9.55,2011-05-14 03:14:20.515884800,4.61,15287.69
min,-80995.00,2010-01-12 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-04 11:36:00,1.25,13953.00
50%,3.00,2011-06-09 10:46:00,2.08,15152.00
75%,10.00,2011-09-06 10:46:00,4.13,16791.00
max,80995.00,2011-12-10 17:19:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


In [8]:
# >'O' is to get the description of the object type columns (categorical data)
df.describe(include='O')

,InvoiceNo,StockCode,Description,Country
count,541909,541909,540455,541909
unique,25900,4070,4223,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,United Kingdom
freq,1114,2313,2369,495478


In [9]:
#finding missing values in the customer id column
df[df["CustomerID"].isna()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-01-12 11:52:00,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,2010-01-12 14:32:00,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,2010-01-12 14:32:00,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,2010-01-12 14:32:00,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,2010-01-12 14:32:00,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,2010-01-12 14:32:00,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,2010-01-12 14:32:00,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,2010-01-12 14:32:00,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,2010-01-12 14:32:00,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,2010-01-12 14:32:00,0.43,NaN,United Kingdom


In [10]:
# Finding Quanity < 0 because that data is not good, as quantity of anything cannot be negative.
df[df["Quantity"]<0].head(10)   

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-01-12 09:41:00,27.50,14527.00,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-01-12 09:49:00,4.65,15311.00,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-01-12 10:24:00,1.65,17548.00,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-01-12 10:24:00,0.29,17548.00,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-01-12 10:24:00,0.29,17548.00,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-01-12 10:24:00,0.29,17548.00,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-01-12 10:24:00,3.45,17548.00,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-01-12 10:24:00,1.65,17548.00,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-01-12 10:24:00,1.65,17548.00,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-01-12 12:38:00,4.25,17897.00,United Kingdom


> We can see that there are some negative quantity values in the dataset. 
> This is likely due to returns or cancellations. 
> I am going to filter out these negative quantity values for the clustering analysis, as they may not represent typical purchasing behavior.

>'C' in the InvoiceNo indicates Cancellation as described by the dataset source.

In [11]:
# display the unique values in the InvoiceNo column to see if there are any values having 6 digits and a C before the 6 digits, which indicates cancellation of the order as described by the dataset source.

df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df[df["InvoiceNo"].str.match("^\\d{6}$") == False]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-01-12 09:41:00,27.50,14527.00,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-01-12 09:49:00,4.65,15311.00,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-01-12 10:24:00,1.65,17548.00,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-01-12 10:24:00,0.29,17548.00,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-01-12 10:24:00,0.29,17548.00,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-09-12 09:57:00,0.83,14397.00,United Kingdom
541541,C581499,M,Manual,-1,2011-09-12 10:28:00,224.69,15498.00,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-09-12 11:57:00,10.95,15311.00,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-09-12 11:58:00,1.25,17315.00,United Kingdom


In [12]:
# as we can see, in the data C is not the only character that is not a digit, there are also some other characters, 
# so we will remove all the characters from the InvoiceNo column to get only the digits,

df["InvoiceNo"].str.replace("[0-9]", "", regex=True).unique()


array(['', 'C', 'A'], dtype=object)

In [13]:
# so we start by check what data contains character A in InvoiceNo column,

df[df["InvoiceNo"].str.startswith("A")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-12-08 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-12-08 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-12-08 14:52:00,-11062.06,NaN,United Kingdom


In [16]:
# stock codes have to be 5 digits, so we will check if there are any stock codes that do not have 5 digits

df["StockCode"] = df["StockCode"].astype(str)
df[(df["StockCode"].str.match("^\\d{5}$") == False) & (df["StockCode"].str.match("^\\d{5}[a-zA-Z]+$") == False)]["StockCode"].unique()

array(['POST', 'D', 'C2', 'DOT', 'M', 'BANK CHARGES', 'S', 'AMAZONFEE',
       'DCGS0076', 'DCGS0003', 'gift_0001_40', 'DCGS0070', 'm',
       'gift_0001_50', 'gift_0001_30', 'gift_0001_20', 'DCGS0055',
       'DCGS0072', 'DCGS0074', 'DCGS0069', 'DCGS0057', 'DCGSSBOY',
       'DCGSSGIRL', 'gift_0001_10', 'PADS', 'DCGS0004', 'DCGS0073',
       'DCGS0071', 'DCGS0068', 'DCGS0067', 'DCGS0066P', 'B', 'CRUK'],
      dtype=object)

as the result of above code snippet give us distinctive list of stock code that does not match our patterns, we will use pattern matching to go through each of this

In [27]:
df[df["StockCode"].str.contains("^P")]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
45,536370,POST,POSTAGE,3,2010-01-12 08:45:00,18.00,12583.00,France
386,536403,POST,POSTAGE,1,2010-01-12 11:27:00,15.00,12791.00,Netherlands
1123,536527,POST,POSTAGE,1,2010-01-12 13:04:00,18.00,12662.00,Germany
5073,536840,POST,POSTAGE,1,2010-02-12 18:27:00,18.00,12738.00,Germany
5258,536852,POST,POSTAGE,1,2010-03-12 09:51:00,18.00,12686.00,France
...,...,...,...,...,...,...,...,...
541198,581493,POST,POSTAGE,1,2011-09-12 10:10:00,15.00,12423.00,Belgium
541216,581494,POST,POSTAGE,2,2011-09-12 10:13:00,18.00,12518.00,Germany
541730,581570,POST,POSTAGE,1,2011-09-12 11:59:00,18.00,12662.00,Germany
541767,581574,POST,POSTAGE,2,2011-09-12 12:09:00,18.00,12526.00,Germany


### Notes

#### Stock Code
* StockCode is meant to follow the pattern `[0-9]{5}` but seems to have legit values for `[0-9]{5}[a-zA-Z]+`
    * Also contains other values:
        | **Code**            | **Description**                                                        | **Action**              |
        |---------------------|------------------------------------------------------------------------|-------------------------|
        | DCGS            | Looks valid, some quantities are negative though and customer ID is null | Exclude from clustering |
        | D               | Looks valid, represents discount values                                | Exclude from clustering |
        | DOT             | Looks valid, represents postage charges                                | Exclude from clustering |
        | M or m          | Looks valid, represents manual transactions                            | Exclude from clustering |
        | C2              | Carriage transaction - not sure what this means                        | Exclude from clustering |
        | BANK CHARGES or B | Bank charges                                                         | Exclude from clustering |
        | S               | Samples sent to customer                                               | Exclude from clustering |
        | CRUK            | CRUK Commission                                                        | Exclude from clustering |
        | gift__XXX       | Purchases with gift cards, might be interesting for another analysis, but no customer data | Exclude |
        | PADS            | Looks like a legit stock code for padding                              | Include                 |
        | AMAZONFEE       | Looks like fees for Amazon shipping or something                       | Exclude for now         |